# Iowa — Title XIII (insurance chapters) → `data/iowa/ins_codes/*.md`

Iowa insurance law lives primarily in **Iowa Code Title XIII — Commerce** under **Chapter 505** onward (official: [legis.iowa.gov — Iowa Code](https://www.legis.iowa.gov/law/iowaCode)). Title XIII also contains **banks, credit unions, and the UCC**, so this notebook **does not** ingest the entire title by default.

It mirrors **Justia**: **[Iowa Code — Title XIII](https://law.justia.com/codes/iowa/title-xiii/)** (`/codes/iowa/title-xiii/…`). Each **chapter** page lists **`section-…`** links (for example [`…/section-515-1/`](https://law.justia.com/codes/iowa/title-xiii/chapter-515/section-515-1/)). **Cloudflare** often blocks plain **`httpx`**; we use **`curl_cffi`** with **`impersonate="chrome120"`**.

**Chapter filter (default):** include a chapter from the Title XIII index when **either** the heading contains **`INSURANCE`**, **`INSURER`**, or **`REINSURANCE`**, **or** the base chapter number is **505–522**, **or** the chapter is **546** (Department of Insurance and Financial Services). Set **`USE_ALL_TITLE_XIII_CHAPTERS = True`** to crawl **every** chapter index under Title XIII (~160 chapters, including non-insurance commerce).

**Discovery:** For each included chapter URL, fetch the chapter index once and collect every **`/section-…`** link.

**Download:** Text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`Iowa_sec_<section>.md`** (hyphens → underscores, e.g. `Iowa_sec_515_1.md`, `Iowa_sec_521a_1.md`).

**Config:** **`CODE_YEAR`** seeds **`/codes/iowa/{year}/title-xiii/`** (normalized to canonical **`/codes/iowa/title-xiii/…`**). **`MAX_SECTIONS`** caps downloads (**0** = all). **`REUSE_DISCOVERED_URLS`** skips rediscovery when **`_iowa_insurance_section_urls.txt`** exists.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/iowa/title-xiii"
CODE_YEAR = "2024"
TITLE_INDEX = f"{BASE}/codes/iowa/{CODE_YEAR}/title-xiii/"

OUT_DIR = Path("data") / "iowa" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
SKIP_EXISTING = True

# If True, every chapter linked from Title XIII is crawled (banks, UCC, etc.).
USE_ALL_TITLE_XIII_CHAPTERS = False

DISCOVERED_LIST = OUT_DIR / "_iowa_insurance_section_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def canon_path(p: str) -> str:
    """Map /codes/iowa/2024/title-xiii/… → /codes/iowa/title-xiii/…"""
    p = p.rstrip("/")
    m = re.match(r"^/codes/iowa/20\d\d(/title-xiii(?:/.*)?)$", p, re.I)
    if m:
        return "/codes/iowa" + m.group(1)
    return p


def chapter_base_num(title: str) -> int | None:
    m = re.match(r"Chapter\s+(\d+)", title.strip(), re.I)
    return int(m.group(1)) if m else None


def include_chapter(title: str) -> bool:
    if USE_ALL_TITLE_XIII_CHAPTERS:
        return True
    u = title.upper()
    if "INSURANCE" in u or "INSURER" in u or "REINSURANCE" in u:
        return True
    n = chapter_base_num(title)
    if n is None:
        return False
    if n == 546:
        return True
    if 505 <= n <= 522:
        return True
    return False


def discover_chapter_urls() -> list[str]:
    html = curl_get(TITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    out: set[str] = set()
    for a in soup.find_all("a", href=True):
        absu = urljoin(TITLE_INDEX, a["href"])
        p = canon_path(path_key(absu))
        if not re.match(r"^/codes/iowa/title-xiii/chapter-[^/]+$", p, re.I):
            continue
        if include_chapter(a.get_text(strip=True)):
            out.add(BASE + p + "/")
    return sorted(out)


def section_urls_for_chapter(ch_url: str) -> list[str]:
    ch_canon = canon_path(path_key(ch_url))
    html = curl_get(BASE + ch_canon + "/")
    soup = BeautifulSoup(html, "html.parser")
    found: set[str] = set()
    for a in soup.find_all("a", href=True):
        absu = urljoin(BASE + ch_canon + "/", a["href"])
        p = canon_path(path_key(absu))
        if "/section-" not in p.lower():
            continue
        if not p.startswith(ch_canon + "/"):
            continue
        found.add(BASE + p + "/")
    return sorted(found)


def discover_all_section_urls() -> list[str]:
    chapters = discover_chapter_urls()
    print(f"Selected {len(chapters)} chapter index pages under Title XIII")
    all_sec: list[str] = []
    for ch in chapters:
        all_sec.extend(section_urls_for_chapter(ch))
    return sorted(all_sec, key=lambda u: section_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    p = canon_path(path_key(url))
    m = re.search(r"/section-([^/]+)/?$", p, re.I)
    if not m:
        raise ValueError(f"cannot parse section id from {url!r}")
    return m.group(1)


def section_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"Iowa_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Iowa Code" in s and "(" in s:
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("IA Code §"):
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_iowa_insurance() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        all_urls = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(all_urls, key=lambda u: section_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        all_urls = discover_all_section_urls()
        print(f"Discovered {len(all_urls)} section URLs")
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"Iowa Code § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Iowa Code — Title XIII (Commerce), insurance-focused chapters**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [legis.iowa.gov — Iowa Code](https://www.legis.iowa.gov/law/iowaCode)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_iowa_insurance()


Selected 95 chapter index pages under Title XIII
Discovered 1514 section URLs
… 200/1514 (wrote=200 skipped=0 failed=0)
… 400/1514 (wrote=400 skipped=0 failed=0)
… 600/1514 (wrote=600 skipped=0 failed=0)
… 800/1514 (wrote=800 skipped=0 failed=0)
… 1000/1514 (wrote=1000 skipped=0 failed=0)
… 1200/1514 (wrote=1200 skipped=0 failed=0)
… 1400/1514 (wrote=1400 skipped=0 failed=0)
Done. wrote=1514 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/iowa/ins_codes


{'wrote': 1514, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
